In [5]:
from pathlib import Path
import json
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

In [6]:
# Choose the input JSON file (JSON Whole Model export)
file_name = "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json"

# Default: workspace-root/JSON Whole Model/<file_name> (works when notebook is in PyDataTransform/)
input_path = (Path('..') / 'JSON Whole Model' / file_name).resolve()
if not input_path.exists():
    input_path = (Path('JSON Whole Model') / file_name).resolve()

assert input_path.exists(), f"File not found: {input_path}"

# 1) Copy the JSON being read into JSON_Edit
json_edit_dir = (Path('..') / 'JSON_Edit').resolve()
if not json_edit_dir.exists():
    json_edit_dir = (Path('JSON_Edit')).resolve()
json_edit_dir.mkdir(parents=True, exist_ok=True)

copied_path = json_edit_dir / input_path.name
copied_path.write_text(input_path.read_text(encoding='utf-8'), encoding='utf-8')
print(f"Copied source JSON to: {copied_path}")

# Keep original JSON structure in memory so we can write it back without escaping '/'
with copied_path.open('r', encoding='utf-8') as f:
    records = json.load(f)

df = pd.DataFrame(records)
working_json_path = copied_path
df.shape

Copied source JSON to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


(180, 4)

#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [7]:
def _extract_guid(props):
    # `props` is the element's `Properties` array.
    # We look for the entry with displayName == 'GUID' (case-insensitive).
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

def _strip_guid_prefix(name):
    if pd.isna(name):
        return name
    text = str(name)
    if '_' not in text:
        return text

    prefix, rest = text.split('_', 1)
    # Remove prefixes like "1JNL9441111_" / "1JNL9442575_" (or similar ID-like prefixes)
    if len(prefix) >= 8 and any(ch.isdigit() for ch in prefix):
        return rest
    return text

def _write_json_unescaped(path, payload):
    # Python json.dumps keeps slashes as '/' (does not force '\\/')
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

# 2 & 3) Clean the Name column by removing GUID-like prefixes
df['Name'] = df['Name'].apply(_strip_guid_prefix)

# Sync cleaned names back to original JSON records
for idx, item in enumerate(records):
    if idx < len(df):
        item['Name'] = None if pd.isna(df.at[idx, 'Name']) else str(df.at[idx, 'Name'])

# Save amended JSON into JSON_Edit (does not change original source file)
_write_json_unescaped(working_json_path, records)
print(f"Updated JSON written to: {working_json_path}")

# Build GUID + key columns (GUID is primary unique key for matching)
df['GUID'] = df['Properties'].apply(_extract_guid)
df['ElementKey'] = df['GUID'].fillna(df['ExternalId'])
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

guid_duplicates = df['GUID'].dropna().duplicated().sum()
print(f"GUID duplicates found: {guid_duplicates}")

table = (
    df[['ElementKey', 'GUID', 'ExternalId', 'Name', 'DbId', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

# 4) Print out the updated table
rows_to_show = 50  # set to None to show all rows (can be slow/huge)
table if rows_to_show is None else table.head(rows_to_show)

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json
GUID duplicates found: 0


,ElementKey,GUID,ExternalId,Name,DbId,NameCount
0,8d3567ff-612b-3878-a628-4b9d838df628,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,1
1,c9232a6d-75c5-3739-a3c2-3777ac67e597,c9232a6d-75c5-3739-a3c2-3777ac67e597,0/0/2/1/2,A-Cable Ladder 90 450,14,3
2,3d931300-8be8-3a23-a464-9cedb7d98460,3d931300-8be8-3a23-a464-9cedb7d98460,0/0/2/1/3,A-Cable Ladder 90 450,15,3
3,19f22aef-af24-3a1c-b63f-1703e94e50ba,19f22aef-af24-3a1c-b63f-1703e94e50ba,0/0/2/1/7,A-Cable Ladder 90 450,19,3
4,167265d4-c3e5-3071-b189-599004f7b9f3,167265d4-c3e5-3071-b189-599004f7b9f3,0/0/2/1/0,A-Cable Ladder MVS,12,9
5,366c5df3-80dd-34c1-95f6-f69c55125c61,366c5df3-80dd-34c1-95f6-f69c55125c61,0/0/2/1/1,A-Cable Ladder MVS,13,9
6,5b00d1af-e807-3f5e-bc90-b4ab1b39b423,5b00d1af-e807-3f5e-bc90-b4ab1b39b423,0/0/2/1/4,A-Cable Ladder MVS,16,9
7,69eed907-29dc-3444-89dd-6e9fe484ce1b,69eed907-29dc-3444-89dd-6e9fe484ce1b,0/0/2/1/5,A-Cable Ladder MVS,17,9
8,97558d3b-c88d-3c4c-b66f-b2109faa27c0,97558d3b-c88d-3c4c-b66f-b2109faa27c0,0/0/2/1/6,A-Cable Ladder MVS,18,9
9,a378d9c0-1349-32e1-9dad-1b5310fafdc9,a378d9c0-1349-32e1-9dad-1b5310fafdc9,0/0/2/1/8,A-Cable Ladder MVS,20,9


In [8]:
# Export the table
out_dir = input_path.parent
# csv_path = out_dir / f"{input_path.stem}_name_table.csv"
xlsx_path = out_dir / f"{input_path.stem}_name_table.xlsx"

# table.to_csv(csv_path, index=False, encoding='utf-8-sig')
# print("Wrote CSV:", csv_path)

# Excel export requires openpyxl (recommended)
try:
    import openpyxl  # noqa: F401
    table.to_excel(xlsx_path, index=False)
    print("Wrote Excel:", xlsx_path)
except ImportError:
    print("Excel export skipped: package 'openpyxl' is not installed.")
    print("Run: pip install openpyxl  (or use notebook package install), then re-run this cell.")

Wrote Excel: C:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002_name_table.xlsx


In [9]:
def _strip_object_prefix(value):
    if not isinstance(value, str):
        return value
    if '_' not in value:
        return value
    prefix, rest = value.split('_', 1)
    if len(prefix) >= 8 and any(ch.isdigit() for ch in prefix):
        return rest
    return value

property_changes = []
objects_changed = 0

for idx, row in df.iterrows():
    object_changed = False
    rec = records[idx] if idx < len(records) else None

    guid = row.get('GUID')
    external_id = row.get('ExternalId')

    cleaned_name = _strip_object_prefix(row.get('Name'))
    if cleaned_name != row.get('Name'):
        df.at[idx, 'Name'] = cleaned_name
        if isinstance(rec, dict):
            rec['Name'] = cleaned_name
        object_changed = True

    props = rec.get('Properties') if isinstance(rec, dict) else row.get('Properties')
    if isinstance(props, list):
        for prop in props:
            if not isinstance(prop, dict):
                continue
            old_value = prop.get('value')
            new_value = _strip_object_prefix(old_value)
            if new_value != old_value:
                prop['value'] = new_value
                object_changed = True
                property_changes.append({
                    'GUID': guid,
                    'ExternalId': external_id,
                    'ObjectName': cleaned_name,
                    'DbId': row.get('DbId'),
                    'Property': prop.get('displayName'),
                    'OldValue': old_value,
                    'NewValue': new_value
                })

    if object_changed:
        objects_changed += 1

# Keep '/' in values such as ExternalId (no escaped '\\/')
_write_json_unescaped(working_json_path, records)
print(f"Updated JSON written to: {working_json_path}")

if property_changes:
    changes_df = pd.DataFrame(property_changes)
    print("\nChanged object-property values (matched by GUID first):")
    display(changes_df[['GUID', 'ExternalId', 'ObjectName', 'DbId', 'Property', 'OldValue', 'NewValue']])
else:
    print("\nNo prefixed property values found.")

print(f"\nTotal objects changed: {objects_changed}")
print(f"Total property values changed: {len(property_changes)}")

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json

Changed object-property values (matched by GUID first):


,GUID,ExternalId,ObjectName,DbId,Property,OldValue,NewValue
0,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,Name,1JNL9322340_A-1JNL9322340 - Pyramid,A-1JNL9322340 - Pyramid
1,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,NAME,1JNL9322340_A-1JNL9322340 - Pyramid,A-1JNL9322340 - Pyramid
2,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,OBJECTTYPE,1JNL9322340_A-1JNL9322340 - Pyramid,A-1JNL9322340 - Pyramid
3,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,GLOBALID,1TQoQ3fqOqqfozt5n_JZ7X,JZ7X
4,8d3567ff-612b-3878-a628-4b9d838df628,0/0/1,A-1JNL9322340 - Pyramid,5,Name,1JNL9322340_A-1JNL9322340 - Pyramid,A-1JNL9322340 - Pyramid
5,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,0/0/2,"A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways",6,Name,"1JNL9362899_A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways","A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways"
6,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,0/0/2,"A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways",6,NAME,"1JNL9362899_A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways","A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways"
7,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,0/0/2,"A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways",6,OBJECTTYPE,"1JNL9362899_A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways","A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways"
8,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,0/0/2,"A-Electrical design requirements, MVS1, NER, Aux Tx-building, Cable ways",6,GLOBALID,1YNBYXeuaxse$_5RIiDq3M,5RIiDq3M
9,c9232a6d-75c5-3739-a3c2-3777ac67e597,0/0/2/1/2,A-Cable Ladder 90 450,14,Name,1JNL9441111_A-Cable Ladder 90 450,A-Cable Ladder 90 450



Total objects changed: 30
Total property values changed: 175
